In [ ]:
import nltk
import string
from datasets import load_dataset
from collections import Counter
from nltk.corpus import stopwords
from nltk.corpus import wordnet
from sklearn.feature_extraction.text import CountVectorizer
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

dataset = load_dataset("emotion")

first_example = dataset["train"][0]
first_text = first_example["text"]

# Шаг 1: Токенизация без доп. очистки
tokens = nltk.word_tokenize(first_text)
print("Посмотрим токены без доп. отчистки")
print(tokens[:100])
print()

all_tokens = []

for i in range(1000):
    text = dataset["train"][i]["text"]
    tokens = nltk.word_tokenize(text)
    all_tokens.extend(tokens)

count = Counter(all_tokens)
print("50 самых популярных токенов")
print(count.most_common(50))
print()

lemmatizer = nltk.WordNetLemmatizer()

stop_words = set(stopwords.words("english"))
new_stops = {'im', 'really', 'little', 'would', 'know'}
stop_words.update(new_stops)

def get_wordnet_pos(tag):
    if tag.startswith('J'): return wordnet.ADJ
    elif tag.startswith('V'): return wordnet.VERB
    elif tag.startswith('R'): return wordnet.ADV
    else: return wordnet.NOUN

def preprocess_with_lemmatization(text):
    text = text.lower()
    tokens = nltk.word_tokenize(text)
    tagged = nltk.pos_tag(tokens)
    #убираем стоп-слова, пунктуацию и слова короче 3 символов ('i', 't', 'am', 'me', 'do', 'up', ...)
    lemmatized_tokens = [
        lemmatizer.lemmatize(token, get_wordnet_pos(tag))
        for token, tag in tagged
        if token.isalpha()
        and token not in stop_words
        and len(token) > 2
    ]
    return lemmatized_tokens

#посмотрим результат на первых 1000 примерах
all_clean_tokens = []
raw_docs = [dataset["train"][i]["text"] for i in range(1000)]

for doc in raw_docs:
    clean_tokens = preprocess_with_lemmatization(doc)
    all_clean_tokens.extend(clean_tokens)

count = Counter(all_clean_tokens)
print("50 самых популярных токенов после очищения")
print(count.most_common(50))

# Склеиваем для CountVectorizer
docs_as_strings = [' '.join(preprocess_with_lemmatization(doc)) for doc in raw_docs]

vectorizer = CountVectorizer(binary=True)
X = vectorizer.fit_transform(docs_as_strings)

print("\nРазмер матрицы после очистки:", X.shape)
print("Размер словаря (стал меньше и качественнее):", len(vectorizer.vocabulary_))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


Посмотрим токены без доп. отчистки
['i', 'didnt', 'feel', 'humiliated']

50 самых популярных токенов
[('i', 1641), ('feel', 736), ('and', 606), ('to', 597), ('the', 520), ('a', 417), ('of', 331), ('that', 328), ('feeling', 296), ('my', 280), ('in', 240), ('it', 185), ('like', 184), ('me', 163), ('have', 161), ('is', 160), ('so', 147), ('was', 143), ('but', 137), ('am', 136), ('im', 135), ('for', 131), ('not', 126), ('with', 119), ('be', 119), ('this', 113), ('about', 104), ('on', 101), ('as', 96), ('or', 94), ('at', 93), ('you', 92), ('when', 91), ('just', 85), ('more', 77), ('can', 72), ('all', 69), ('if', 69), ('because', 68), ('are', 68), ('do', 67), ('really', 64), ('t', 62), ('little', 61), ('up', 58), ('by', 55), ('very', 55), ('out', 53), ('would', 53), ('know', 52)]

50 самых популярных токенов после очищения
[('like', 189), ('make', 81), ('get', 72), ('time', 61), ('want', 60), ('love', 57), ('life', 53), ('thing', 48), ('think', 47), ('people', 45), ('day', 41), ('something',

In [ ]:
#оставим сущ и прил
def filter_nouns_adjs(text):
    text = text.lower()
    tokens = nltk.word_tokenize(text)
    tagged = nltk.pos_tag(tokens)
    #N - сущ, J - прил
    result = [
        lemmatizer.lemmatize(token, get_wordnet_pos(tag))
        for token, tag in tagged
        if (tag.startswith('N') or tag.startswith('J'))
        and token.isalpha()
        and token not in stop_words
    ]
    return result

#сущ + прил + гл
def filter_nouns_adjs_verbs(text):
    text = text.lower()
    tokens = nltk.word_tokenize(text)
    tagged = nltk.pos_tag(tokens)
    # N - сущ, J - прил, V - гл
    result = [
        lemmatizer.lemmatize(token, get_wordnet_pos(tag))
        for token, tag in tagged
        if (tag.startswith('N') or tag.startswith('J') or tag.startswith('V'))
        and token.isalpha()
        and token not in stop_words
    ]
    return result

print("\nСравнение результатов:")
for i in range(10):
    t = dataset["train"][i]["text"]
    print(f"{i+1}. Сущ + Прил: {' '.join(filter_nouns_adjs(t))}")
    print(f"   Сущ + Прил + Гл: {' '.join(filter_nouns_adjs_verbs(t))}")


Сравнение результатов:
1. Сущ + Прил: 
   Сущ + Прил + Гл: didnt humiliate
2. Сущ + Прил: hopeless damned hopeful someone awake
   Сущ + Прил + Гл: go hopeless damned hopeful someone care awake
3. Сущ + Прил: minute greedy wrong
   Сущ + Прил + Гл: grab minute post greedy wrong
4. Сущ + Прил: nostalgic fireplace property
   Сущ + Прил + Гл: nostalgic fireplace property
5. Сущ + Прил: grouchy
   Сущ + Прил + Гл: grouchy
6. Сущ + Прил: ive burdened wasnt sure
   Сущ + Прил + Гл: ive burdened wasnt sure
7. Сущ + Прил: ive milligram time recommended amount ive lot funny
   Сущ + Прил + Гл: ive take milligram time recommended amount ive fall asleep lot funny
8. Сущ + Прил: life teenager year old man
   Сущ + Прил + Гл: confuse life teenager jade year old man
9. Сущ + Прил: petronas year petronas huge profit
   Сущ + Прил + Гл: petronas year petronas perform make huge profit
10. Сущ + Прил: romantic
   Сущ + Прил + Гл: romantic


In [ ]:
# Загружаем датасет
dataset = load_dataset("emotion")

# Берём первый пример из обучающей выборки
first_example = dataset["train"][0]
first_text = first_example["text"]

# Шаг 1: Токенизация без доп. очистки
tokens = nltk.word_tokenize(first_text)

lemmatizer = nltk.WordNetLemmatizer()

def get_wordnet_pos(tag):
  if tag.startswith('J'): return wordnet.ADJ
  elif tag.startswith('V'): return wordnet.VERB
  elif tag.startswith('R'): return wordnet.ADV
  else: return wordnet.NOUN

def preprocess_with_lemmatization(text):
  text = text.lower()
  tokens = nltk.word_tokenize(text)
  tagged = nltk.pos_tag(tokens)
  lemmatized_tokens = [lemmatizer.lemmatize(token, get_wordnet_pos(tag)) for token, tag in tagged if token not in string.punctuation and token not in stop_words]
  return lemmatized_tokens

#Вариант 2 — на вход сырые строки, CountVectorizer сам вызывает нашу функцию лемматизации
#(в ней уже есть приведение к нижнему регистпу, поэтому повторно не надо и в параметрах стоит lowercase=False):
vectorizer = CountVectorizer(binary=True, tokenizer=preprocess_with_lemmatization, lowercase=False, token_pattern=None)
X = vectorizer.fit_transform(raw_docs)

print("Размер матрицы:", X.shape)
print("Размер словаря:", len(vectorizer.vocabulary_))

Размер матрицы: (1000, 2755)
Размер словаря: 2755


In [ ]:
# Загружаем датасет ag_news
dataset = load_dataset("emotion")

# Берём первый пример из обучающей выборки
first_example = dataset["train"][0]
first_text = first_example["text"]

# Шаг 1: Токенизация без доп. очистки
tokens = nltk.word_tokenize(first_text)

lemmatizer = nltk.WordNetLemmatizer()

def get_wordnet_pos(tag):
  if tag.startswith('J'): return wordnet.ADJ
  elif tag.startswith('V'): return wordnet.VERB
  elif tag.startswith('R'): return wordnet.ADV
  else: return wordnet.NOUN

def preprocess_with_lemmatization(text):
  text = text.lower()
  tokens = nltk.word_tokenize(text)
  tagged = nltk.pos_tag(tokens)
  lemmatized_tokens = [lemmatizer.lemmatize(token, get_wordnet_pos(tag)) for token, tag in tagged if token not in string.punctuation and token not in stop_words]
  return lemmatized_tokens

#Вариант 3 — на вход сырые строки, без лемматизации (CountVectorizer сам токенизирует, только lowercase и разбивка по регулярному выражению):
vectorizer = CountVectorizer(binary=True)
X = vectorizer.fit_transform(raw_docs)

print("Размер матрицы:", X.shape)
print("Размер словаря:", len(vectorizer.vocabulary_))

Размер матрицы: (1000, 3307)
Размер словаря: 3307


In [ ]:
text = "Hello, world!"
rez = nltk.word_tokenize(text)
print(rez)

['Hello', ',', 'world', '!']


проверили что знаки не удаляются

In [ ]:
#удалим знаки до
text = text.translate(str.maketrans('', '', string.punctuation))
rez = nltk.word_tokenize(text)
print(rez)

['Hello', 'world']


In [ ]:
#удалим знаки после
text = "Hello, world!"
rez = nltk.word_tokenize(text)
rez = [i for i in rez if i not in string.punctuation]
print(rez)

['Hello', 'world']


In [ ]:
import nltk
import string
from datasets import load_dataset

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

from nltk.corpus import stopwords
from nltk.corpus import wordnet
from sklearn.feature_extraction.text import CountVectorizer

stop_words = set(stopwords.words("english"))

dataset = load_dataset("SetFit/20_newsgroups")

classes = [
    "comp.sys.ibm.pc.hardware",
    "comp.sys.mac.hardware",
    "comp.graphics",
    "comp.windows.x"
]

dataset_new = dataset["train"].filter(
    lambda example: example["label_text"] in classes
)

first_example = dataset_new[0]
first_text = first_example["text"]


# Шаг 1: Токенизация без доп. очистки
tokens = nltk.word_tokenize(first_text)
print("Посмотрим токены без доп. отчистки")
print(tokens[:100])
print()

all_tokens = []

for i in range(1000):
    text = dataset["train"][i]["text"]
    tokens = nltk.word_tokenize(text)
    all_tokens.extend(tokens)

count = Counter(all_tokens)
print("50 самых популярных токенов")
print(count.most_common(50))
print()

lemmatizer = nltk.WordNetLemmatizer()

stop_words = set(stopwords.words("english"))

def get_wordnet_pos(tag):
    if tag.startswith('J'): return wordnet.ADJ
    elif tag.startswith('V'): return wordnet.VERB
    elif tag.startswith('R'): return wordnet.ADV
    else: return wordnet.NOUN

def preprocess_with_lemmatization(text):
    text = text.lower()
    tokens = nltk.word_tokenize(text)
    tagged = nltk.pos_tag(tokens)
    #убираем стоп-слова, пунктуацию и слова короче 3 символов ('i', 't', 'am', 'me', 'do', 'up', ...)
    lemmatized_tokens = [
        lemmatizer.lemmatize(token, get_wordnet_pos(tag))
        for token, tag in tagged
        if token.isalpha()
        and token not in stop_words
        and len(token) > 2
    ]
    return lemmatized_tokens

#посмотрим результат на первых 1000 примерах
all_clean_tokens = []
raw_docs = [dataset["train"][i]["text"] for i in range(1000)]

for doc in raw_docs:
    clean_tokens = preprocess_with_lemmatization(doc)
    all_clean_tokens.extend(clean_tokens)

count = Counter(all_clean_tokens)
print("50 самых популярных токенов после очищения")
print(count.most_common(50))

# Склеиваем для CountVectorizer
docs_as_strings = [' '.join(preprocess_with_lemmatization(doc)) for doc in raw_docs]

vectorizer = CountVectorizer(binary=True)
X = vectorizer.fit_transform(docs_as_strings)

print("\nРазмер матрицы после очистки:", X.shape)
print("Размер словаря (стал меньше и качественнее):", len(vectorizer.vocabulary_))

#оставим сущ и прил
def filter_nouns_adjs(text):
    text = text.lower()
    tokens = nltk.word_tokenize(text)
    tagged = nltk.pos_tag(tokens)
    #N - сущ, J - прил
    result = [
        lemmatizer.lemmatize(token, get_wordnet_pos(tag))
        for token, tag in tagged
        if (tag.startswith('N') or tag.startswith('J'))
        and token.isalpha()
        and token not in stop_words
    ]
    return result

#сущ + прил + гл
def filter_nouns_adjs_verbs(text):
    text = text.lower()
    tokens = nltk.word_tokenize(text)
    tagged = nltk.pos_tag(tokens)
    # N - сущ, J - прил, V - гл
    result = [
        lemmatizer.lemmatize(token, get_wordnet_pos(tag))
        for token, tag in tagged
        if (tag.startswith('N') or tag.startswith('J') or tag.startswith('V'))
        and token.isalpha()
        and token not in stop_words
    ]
    return result

print("\nСравнение результатов:")
for i in range(5):
    text = dataset["train"][i]["text"]
    print(f"{i+1}. Сущ + Прил: {' '.join(filter_nouns_adjs(text))}")
    print(f"   Сущ + Прил + Гл: {' '.join(filter_nouns_adjs_verbs(text))}")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


README.md:   0%|          | 0.00/734 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.jsonl:   0%|          | 0.00/14.8M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/8.91M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/11314 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7532 [00:00<?, ? examples/s]

Filter:   0%|          | 0/11314 [00:00<?, ? examples/s]

Посмотрим токены без доп. отчистки
['A', 'fair', 'number', 'of', 'brave', 'souls', 'who', 'upgraded', 'their', 'SI', 'clock', 'oscillator', 'have', 'shared', 'their', 'experiences', 'for', 'this', 'poll', '.', 'Please', 'send', 'a', 'brief', 'message', 'detailing', 'your', 'experiences', 'with', 'the', 'procedure', '.', 'Top', 'speed', 'attained', ',', 'CPU', 'rated', 'speed', ',', 'add', 'on', 'cards', 'and', 'adapters', ',', 'heat', 'sinks', ',', 'hour', 'of', 'usage', 'per', 'day', ',', 'floppy', 'disk', 'functionality', 'with', '800', 'and', '1.4', 'm', 'floppies', 'are', 'especially', 'requested', '.', 'I', 'will', 'be', 'summarizing', 'in', 'the', 'next', 'two', 'days', ',', 'so', 'please', 'add', 'to', 'the', 'network', 'knowledge', 'base', 'if', 'you', 'have', 'done', 'the', 'clock', 'upgrade', 'and', 'have', "n't", 'answered', 'this', 'poll', '.']

50 самых популярных токенов
[(',', 10327), ('.', 9054), ('the', 8354), ('>', 6922), ("'AX", 4866), ('--', 4576), ('to', 4307), ('o